# Structured Output 

Models can be requested to provide their response in a format matchig a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output 


# Pydantic 

Pydantic model provides the richest feature set with field validation , descriptions and nested structures

In [8]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

 

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C8997EB980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C899BE5550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import BaseModel , Field

class movie(BaseModel):
    title:str= Field (description="Title of the movie")   #if the value of title was int , it will give an error .Because of field validation , always give appropriate value to the variable 
    year:int =Field(description="This year movie was released")
    director:str=Field(description="the director of the movie")
    rating:float = Field(description="The movies rating out of 10")
    

In [10]:
model_with_structure=model.with_structured_output(movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C8997EB980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C899BE5550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': 

In [11]:
model.invoke("provide details about the movie Inception")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let\'s start by recalling what I know about it. Directed by Christopher Nolan, it\'s a sci-fi thriller released in 2010. Leonardo DiCaprio stars as Dominic Cobb, a thief who steals secrets by infiltrating the subconscious. The main plot is about using the concept of shared dreaming to plant an idea into someone\'s mind, which is the opposite of theft. \n\nThe main characters include Ariadne (Elliot Page), who is a young architect trained to create dreams, Arthur (Joseph Gordon-Levitt), the team\'s forger, Eames (Tom Hardy), a master of disguises, and Saito (Ken Watanabe). The antagonist is Robert Fischer (Cillian Murphy), the heir to a corporation. \n\nThe story has multiple layers of dreams with different time dilations. The deeper they go into the dream, the slower time becomes. There\'s a concept called limbo, which is a dream within a dream, where people can get stuck for a long time. \n\nThe plo

In [13]:
model_with_structure.invoke("provide details about the movie Inception") # output provided will be in the defined structured format 

movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

# Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel , Field

class movie(BaseModel):
    """A movie with details"""
    title:str= Field (...,description="Title of the movie")   #if the value of title was int , it will give an error .Because of field validation , always give appropriate value to the variable 
    year:int =Field(...,description="This year movie was released")
    director:str=Field(...,description="the director of the movie")
    rating:float = Field(...,description="The movies rating out of 10")

model_with_structure = model.with_structured_output(movie,include_raw =True) # include_raw =True gives the raw message of AImessage

In [17]:
model_with_structure.invoke("provide details about the movie Inception")


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a function called movie that requires title, year, director, and rating. I need to make sure I have all those parameters. I know Inception was directed by Christopher Nolan. The release year was 2010. The rating is a bit tricky; I remember it has a high rating on IMDb, maybe around 8.8. Let me confirm that. Yes, IMDb does have it at 8.8. So I should structure the tool call with these details. Let me double-check each required field to ensure none are missing. Title: Inception, Year: 2010, Director: Christopher Nolan, Rating: 8.8. All required parameters are present. Alright, time to format that into the JSON object as specified.\n", 'tool_calls': [{'id': 'yvw17ewgw', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'movie'}, 'type': 'function'}]}, 

# Nested Structure

In [21]:
from pydantic import BaseModel ,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title : str
    year : int
    cast : list[Actor]
    genre: list[str]
    budget: float | None = Field(None , description ="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Joshua')], genre=['Sci-Fi', 'Action', 'Thriller'], budget=160.0)

# TypedDict

TypedDict provides a simpler alternative using Python's built-in typing. ideal when you don't need runtime validation 

In [26]:
from typing_extensions import TypedDict, Annotated

class movieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"Title of the movie"]   #if the value of title was int , it will give an error .Because of field validation , always give appropriate value to the variable 
    year:Annotated[str,...,"This year movie was released"]
    director:Annotated[str,...,"the director of the movie"]
    rating:Annotated[str,...,"The movies rating out of 10"]

model_withtypeddict = model.with_structured_output(movieDict)
response= model_withtypeddict.invoke("please provide the details of the movie Avengers")
response
    

{'director': 'Joss Whedon',
 'rating': '8.1',
 'title': 'Avengers',
 'year': '2012'}

In [27]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title : str
    year : int
    cast : list[Actor]
    genre: list[str]
    budget: float | None = Field(None , description ="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Antagonist'}],
 'genre': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Inception',
 'year': 2010}

In [28]:
model_withtypeddict.profile


AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [30]:
model.profile  # gives the tools / fetaures of the model is supporting(qwen3 here)

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

# DataClasses

A data class is a class typically containing mainly data , although there arent really any restrictions. You ccreate it using the @dataclass decorator 

In [ ]:
import os 
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")




In [35]:
from pydantic import BaseModel , Field
from langchain.agents import create_agent

class contactinfo(BaseModel):
    """contact information for a prerson"""
    name:str = Field(description = "The name of the person")
    email:str = Field(description="The email address of the person ")
    phone:str = Field(description="the phoner number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format= contactinfo  # auto-selects providerstrategy
)

result = agent.invoke({
    "messages": [{"role":"user" , "content": "Extract contact info from : john doe, john@example.com,(555) 123-4567"}]
})

print(result["structured_response"])


name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [ ]:
## TypeDict

from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class contactinfo(TypedDict):
    """contact information for a prerson"""
    name:str #  "The name of the person"
    email:str # "The email address of the person "
    phone:str # "the phoner number of the person"

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format= contactinfo  # auto-selects providerstrategy
)

result = agent.invoke({
    "messages": [{"role":"user" , "content": "Extract contact info from : john doe, john@example.com,(555) 123-4567"}]
})

print(result)
print(result["structured_response"])


{'messages': [HumanMessage(content='Extract contact info from : john doe, john@example.com,(555) 123-4567', additional_kwargs={}, response_metadata={}, id='4df3f003-0102-47c9-8bb2-94fe240a75d6'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given string: "john doe, john@example.com,(555) 123-4567". Let me break this down.\n\nFirst, I need to identify the name, email, and phone number. The name is "john doe". I should capitalize it properly as "John Doe" maybe, but the function doesn\'t specify formatting, so perhaps just split the first and last name. The email is clearly "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number needs any formatting, like removing parentheses or dashes, but the function parameters just require a string. So I can input it as is.\n\nThe function requires the parameters name, email, and phone. All three are present here. So I need to struct

In [41]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class contactinfo:
   """contact information for a prerson"""
   name:str
   email:str
   phone:str

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format= contactinfo  # auto-selects providerstrategy
)

result = agent.invoke({
    "messages": [{"role":"user" , "content": "Extract contact info from : john doe, john@example.com,(555) 123-4567"}]
})

print(result)
print(result["structured_response"])

    

{'messages': [HumanMessage(content='Extract contact info from : john doe, john@example.com,(555) 123-4567', additional_kwargs={}, response_metadata={}, id='8f722e7d-07bf-4ad4-9bdd-2f60bb2ed094'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "john doe, john@example.com,(555) 123-4567". The function provided is contactinfo, which requires name, email, and phone. \n\nFirst, I need to parse the name. The text starts with "john doe", which is likely the full name. The email is clearly "john@example.com". The phone number is "(555) 123-4567". I should check if all required fields are present. The name is there, email and phone too. The parameters are all required, so no missing data. I just need to format them into the function\'s parameters. Make sure the name is properly capitalized? The input has "john doe" in lowercase, but maybe it\'s better to present it as "John Doe". However, the